# Data Preprocessing 

This notebook prepares the dataset that will be used throughout the project. 
The main dataset is the New York City yellow taxi trip data, which is combined 
with weather data as the external dataset.

The yellow taxi trip dataset is cleaned and prepared by filtering to the six-month 
period of interest, performing data cleaning and validation steps, creating 
variables required for the analysis and retaining only the relevant columns. 
Similar preprocessing steps are performed for the external weather dataset.

The processed taxi and weather data are then joined so that each taxi 
trip can be associated with the weather conditions at the time of its pickup.

The output of this notebook is the taxi-weather dataset, which is saved for the subsequent feature engineering and aggregation steps.

In [1]:
# Importing the libraries required throughout the notebook
from pathlib import Path
import glob
import os
import urllib.request
import zipfile
import shutil

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import pandas as pd
import numpy as np
import geopandas as gpd
from sklearn.neighbors import BallTree

# Creating the directory to store processed datasets
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# Initialising the Spark session for preprocessing
spark = (
    SparkSession.builder
    .appName("MAST30034_Project1_Preprocessing")
    .config("spark.driver.memory", "1500m")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

def shape_change(step_name, before_count, df, save_step=False, save_name=None):

    """ 
    This function reports the change in row count after a preprocessing step, the current column count 
    and optionally saves the resulting DataFrame as a Parquet file to reduce memory usage

    Parameters: 
    step_name: Name of the preprocessing step
    before_count: Number of rows in the DataFrame before applying the preprocessing step
    df: PySpark DataFrame after the preprocessing step
    save_step: Whether to save the resulting DataFrame as Parquet or not (optional)
    save_name: Filename used when saving DataFrame as a Parquet file 

    Returns:
    The resulting rows and DataFrame after the preprocessing step

    """

    after_count = df.count()
    after_columns = len(df.columns) 
    removed = before_count - after_count
    percentage_removed = (removed / before_count * 100) if before_count else 0

    if save_step:
        path = processed_dir / f"{save_name}.parquet"
        df.write.mode("overwrite").parquet(str(path))
        df = spark.read.parquet(str(path))

    print(
        f"{step_name}\n"
        f"  Before:  {before_count:,}\n"
        f"  After:   {after_count:,}\n"
        f"  Removed: {removed:,} ({percentage_removed:.2f}%)\n"
        f"  Columns: {after_columns}\n"
    )

    return after_count, df

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/31 18:00:44 WARN Utils: Your hostname, MAHIKA-F8R8E40, resolves to a loopback address: 127.0.1.1; using 192.168.21.80 instead (on interface eth0)
26/08/31 18:00:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/31 18:00:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/31 18:00:54 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## 1. Yellow Taxi Dataset

In [2]:
# Loading the six months of taxi trip data required for the analysis
MONTHS = ["2023-11", "2023-12", "2024-01", "2024-02", "2024-03", "2024-04"]

taxi_urls = [ f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{month}.parquet" 
             for month in MONTHS ]

taxi_paths = [ f"../data/raw/yellow_tripdata_{month}.parquet" 
              for month in MONTHS ]

for url, path in zip(taxi_urls, taxi_paths): 
    if not os.path.exists(path): 
        print(f"Downloading {os.path.basename(path)}") 
        urllib.request.urlretrieve(url, path)

# Loading the taxi files into a single Spark DataFrame
taxi_df = spark.read.parquet(*taxi_paths)

# Checking the structure and size of the raw dataset before preprocessing
print("Raw taxi data schema:")
taxi_df.printSchema()

raw_taxi_count = taxi_df.count()
print(f"Raw taxi data rows: {raw_taxi_count:,}")

Raw taxi data schema:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



Raw taxi data rows: 19,785,349


In [3]:
# Filtering the dataset to keep only trips using the standard metered fare
before_count = taxi_df.count()

taxi_df = taxi_df.filter(F.col("RatecodeID") == 1)

before_count, taxi_df = shape_change("Filtering to standard rate", before_count, taxi_df)

Filtering to standard rate
  Before:  19,785,349
  After:   17,240,680
  Removed: 2,544,669 (12.86%)
  Columns: 19



In [4]:
# Removing zones 264 and 265 as they are listed as  "Unknown" and
# "Outside of NYC" in the Taxi Zone Lookup Table, respectively
taxi_df = taxi_df.filter(~F.col("PULocationID").isin(264, 265))
                                                           
before_count, taxi_df = shape_change("Removing non-geographic zones",  
    before_count, taxi_df)

Removing non-geographic zones
  Before:  17,240,680
  After:   17,164,542
  Removed: 76,138 (0.44%)
  Columns: 19



In [5]:
# Performing data quality checks to identify potential anomalies

# Checking the ranges of pickup and dropoff times, trip distance and fare amount
taxi_df.select(
    F.min("tpep_pickup_datetime").alias("earliest_pickup"),
    F.max("tpep_pickup_datetime").alias("latest_pickup")
).show()

taxi_df.select(
    F.min("trip_distance").alias("min_distance"),
    F.max("trip_distance").alias("max_distance"),
    F.min("fare_amount").alias("min_fare"),
    F.max("fare_amount").alias("max_fare"),
    F.min("tpep_dropoff_datetime").alias("earliest_dropoff"),
    F.max("tpep_dropoff_datetime").alias("latest_dropoff")
).show()

# Calculating trip duration (in minutes)
taxi_df = taxi_df.withColumn(
    "trip_duration_min",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
)

# Checking range of the calculated trip durations 
taxi_df.select(
    F.min("trip_duration_min").alias("min_duration_min"),
    F.max("trip_duration_min").alias("max_duration_min")
).show()

# Checking for non-positive distances, fares or durations and invalid
# pickup and dropoff timestamp ordering
taxi_df.select(
    F.sum(F.when(F.col("trip_distance") <= 0, 1).otherwise(0)).alias("non_positive_distance"),
    F.sum(F.when(F.col("fare_amount") <= 0, 1).otherwise(0)).alias("non_positive_fare"),
    F.sum(F.when(F.col("trip_duration_min") <= 0, 1).otherwise(0)).alias("non_positive_duration"),
    F.sum(F.when(F.col("tpep_dropoff_datetime") <= F.col("tpep_pickup_datetime"), 1
                 ).otherwise(0)).alias("invalid_time_order")
).show()

+-------------------+-------------------+
|    earliest_pickup|      latest_pickup|
+-------------------+-------------------+
|2002-12-31 22:10:04|2024-05-01 00:02:36|
+-------------------+-------------------+



+------------+------------+--------+--------+-------------------+-------------------+
|min_distance|max_distance|min_fare|max_fare|   earliest_dropoff|     latest_dropoff|
+------------+------------+--------+--------+-------------------+-------------------+
|         0.0|     9211.95| -1087.3| 2320.11|2002-12-31 22:14:22|2024-05-01 23:22:35|
+------------+------------+--------+--------+-------------------+-------------------+



+----------------+----------------+
|min_duration_min|max_duration_min|
+----------------+----------------+
|          -58.55|          9455.4|
+----------------+----------------+



+---------------------+-----------------+---------------------+------------------+
|non_positive_distance|non_positive_fare|non_positive_duration|invalid_time_order|
+---------------------+-----------------+---------------------+------------------+
|               133062|           213061|                 4294|              4294|
+---------------------+-----------------+---------------------+------------------+



In [6]:
# Filtering trips to the analysis period from November 2023 to April 2024
START_DATE = "2023-11-01"
END_DATE = "2024-05-01"

taxi_df = taxi_df.filter(
    (F.col("tpep_pickup_datetime") >= F.lit(START_DATE)) &
    (F.col("tpep_pickup_datetime") < F.lit(END_DATE))
)

before_count, taxi_df = shape_change("Filtering to analysis period", before_count, taxi_df)

Filtering to analysis period
  Before:  17,164,542
  After:   17,164,500
  Removed: 42 (0.00%)
  Columns: 20



In [7]:
# Removing invalid trips with non-positive distance, fare or duration
taxi_df = taxi_df.filter(
    (F.col("trip_distance") > 0) &
    (F.col("fare_amount") > 0) &
    (F.col("trip_duration_min") > 0)
)

before_count, taxi_df = shape_change("Removing invalid trips", before_count, taxi_df)

Removing invalid trips
  Before:  17,164,500
  After:   16,833,139
  Removed: 331,361 (1.93%)
  Columns: 20



In [8]:
# Checking for missing values in the key features required for the analysis
taxi_df.select(
    F.sum(F.col("tpep_pickup_datetime").isNull().cast("int")).alias("missing_pickup_datetime"),

    F.sum(F.col("tpep_dropoff_datetime").isNull().cast("int")).alias("missing_dropoff_datetime"),

    F.sum(F.col("trip_distance").isNull().cast("int")).alias("missing_distance"),

    F.sum(F.col("fare_amount").isNull().cast("int")).alias("missing_fare")
).show()

+-----------------------+------------------------+----------------+------------+
|missing_pickup_datetime|missing_dropoff_datetime|missing_distance|missing_fare|
+-----------------------+------------------------+----------------+------------+
|                      0|                       0|               0|           0|
+-----------------------+------------------------+----------------+------------+



In [9]:
# Inspecting the distribution of trip duration to identify unusually long trips
taxi_df.select("trip_duration_min").summary("min", "50%", "90%", "99%", "99.9%", "max").show()

# Removing trips longer than 90 minutes 
taxi_df = taxi_df.filter(
    F.col("trip_duration_min") <= 90
)

before_count, taxi_df = shape_change("Removing trips longer than 90 minutes", before_count, taxi_df)

+-------+--------------------+
|summary|   trip_duration_min|
+-------+--------------------+
|    min|0.016666666666666666|
|    50%|               11.95|
|    90%|               27.55|
|    99%|   50.68333333333333|
|  99.9%|               88.25|
|    max|              9455.4|
+-------+--------------------+



Removing trips longer than 90 minutes
  Before:  16,833,139
  After:   16,816,649
  Removed: 16,490 (0.10%)
  Columns: 20



In [10]:
# Calculating the average trip speed in miles per hour
taxi_df = taxi_df.withColumn(
    "trip_speed_mph", F.col("trip_distance") / (F.col("trip_duration_min") / 60)
)

# Inspecting the distribution of average trip speed to identify implausibly high-speed trips
taxi_df.select("trip_speed_mph").summary("min", "50%", "90%", "95%", "99%", "99.9%", "max").show()

# Removing trips with an average speed above 45 mph
taxi_df = taxi_df.filter(F.col("trip_speed_mph") <= 45)

before_count, taxi_df = shape_change("Removing trips with average speed above 45 mph", before_count, taxi_df)

+-------+--------------------+
|summary|      trip_speed_mph|
+-------+--------------------+
|    min|0.009283135636926251|
|    50%|   9.111864406779663|
|    90%|   17.08769106999196|
|    95%|  21.795637198622273|
|    99%|    32.0531561461794|
|  99.9%|  44.188491164476666|
|    max|             64080.0|
+-------+--------------------+



Removing trips with average speed above 45 mph
  Before:  16,816,649
  After:   16,801,836
  Removed: 14,813 (0.09%)
  Columns: 21



In [11]:
# Removing three implausible records with extreme fares relative
# to their trip distance and duration

taxi_df = taxi_df.filter(
    ~(
        ((F.col("fare_amount") == 2320.11) &
         (F.col("trip_distance") == 6.70) &
         (F.col("trip_duration_min") == 27.033333333333335))
        |
        ((F.col("fare_amount") == 680.00) &
         (F.col("trip_distance") == 0.47) &
         (F.col("trip_duration_min") == 2.0))
        |
        ((F.col("fare_amount") == 243.75) &
         (F.col("trip_distance") == 0.47) &
         (F.col("trip_duration_min") == 5.333333333333333))
    )
)

before_count, taxi_df = shape_change("Removing identified extreme-fare records", 
                            before_count, taxi_df, save_step=True, save_name="taxidf_after_filtering")

Removing identified extreme-fare records
  Before:  16,801,836
  After:   16,801,833
  Removed: 3 (0.00%)
  Columns: 21



In [12]:
# Removing exact duplicate trip records to prevent the same trip
# from being counted more than once
taxi_df = taxi_df.dropDuplicates()   

final_count, taxi_df = shape_change("Removing exact duplicate rows", before_count, taxi_df, save_step=True, save_name="taxidf_no_duplicates")

Removing exact duplicate rows
  Before:  16,801,833
  After:   16,801,832
  Removed: 1 (0.00%)
  Columns: 21



In [13]:
# Loading the Taxi Zone Lookup Table to validate the pickup
# location IDs present in the taxi dataset
zones_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
zones_path = "../data/raw/taxi_zone_lookup.csv"

if not os.path.exists(zones_path):
    print("Downloading taxi zone lookup table")
    urllib.request.urlretrieve(zones_url, zones_path)

zones_df = spark.read.csv(zones_path, header=True, inferSchema=True)

valid_zone_ids = zones_df.select("LocationID").distinct()

# Identifying taxi records whose pickup location ID does not appear
# in the Taxi Zone Lookup Table
invalid_pickup_id = taxi_df.join(
    valid_zone_ids, taxi_df.PULocationID == valid_zone_ids.LocationID, 
    "left_anti"
    )

invalid_pickup_count = invalid_pickup_id.count()

print(f"Number of rows with Invalid pickup ID: {invalid_pickup_count:,}")

Number of rows with Invalid pickup ID: 0


In [14]:
# Extracting useful components from the pickup timestamp for easier analysis

taxi_df = (
    taxi_df
    .withColumn("week_day", F.dayofweek(F.col("tpep_pickup_datetime")))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumn(
        "time_of_day",
        F.when((F.col("pickup_hour") >= 5)  & (F.col("pickup_hour") < 8),  "early_morning")
         .when((F.col("pickup_hour") >= 8)  & (F.col("pickup_hour") < 12), "morning")
         .when((F.col("pickup_hour") >= 12) & (F.col("pickup_hour") < 16), "afternoon")
         .when((F.col("pickup_hour") >= 16) & (F.col("pickup_hour") < 20), "evening")
         .when((F.col("pickup_hour") >= 20) & (F.col("pickup_hour") < 23), "night")
         .otherwise("late_night")   
    )
)

In [15]:
# Retaining only the columns required for subsequent steps

kept_columns = ["pickup_date", "pickup_hour", "time_of_day", "week_day", "trip_duration_min",  
                "trip_distance", "trip_speed_mph", "fare_amount", "PULocationID"]

taxi_df = taxi_df.select(kept_columns)

In [16]:
# Saving the cleaned taxi dataset

final_path = processed_dir / "taxi_df.parquet"
taxi_df.write.mode("overwrite").parquet(str(final_path))

taxi_df = spark.read.parquet(str(final_path))

print(f"Rows:    {taxi_df.count():,}")
print(f"Columns: {len(taxi_df.columns)}")

Rows:    16,801,832
Columns: 9


In [17]:
# Deleting intermediate parquet files created by the shape_change function

temp_files = [
    processed_dir / "taxidf_after_filtering.parquet",
    processed_dir / "taxidf_no_duplicates.parquet"
]

for file in temp_files:
    if file.exists():
        shutil.rmtree(file)
        print(f"Deleted: {file}")

Deleted: ../data/processed/taxidf_after_filtering.parquet
Deleted: ../data/processed/taxidf_no_duplicates.parquet


## 2. External Weather Dataset 

In [18]:
def classify_precip_type(weather_type_str):

    """ 
    This function classifies the detailed weather condition codes in the HourlyPresentWeatherType column 
    into broader precipitation categories

    Parameters:
    weather_type_str: NOAA weather condition code to be classified
    
    Returns:
    The corresponding precipitation category
    
    """

    if pd.isna(weather_type_str) or str(weather_type_str).strip() == "":
        return "no_prec"      # Missing weather codes are treated as no precipitation
    s = str(weather_type_str).upper()
    if "TS" in s:
        return "thunderstorm"
    elif "FZRA" in s or "FZDZ" in s:
        return "freezing_rain"
    elif "GR" in s or "GS" in s:
        return "hail"
    elif "SN" in s or "SG" in s or "IC" in s or "PL" in s:
        return "snow"
    elif "RA" in s or "DZ" in s:
        return "rain"
    else:
        return "no_prec"  # Conditions that do not involve any form of precipitation


In [19]:
def load_weather_data(filepath):

    """
    This function loads and preprocesses a raw weather data file by selecting the
    required columns, converting features to the appropriate data types, retaining
    only hourly observations and extracting useful weather variables

    Parameters:
    filepath: Path to the raw weather data file.

    Returns:
    An initially processed weather DataFrame.
    """
     
    columns_needed = ["STATION", "NAME", "LATITUDE", "LONGITUDE", "DATE", "REPORT_TYPE",
        "HourlyPrecipitation", "HourlyDryBulbTemperature", "HourlyPresentWeatherType", "HourlyWindSpeed"
        ]

    weather_df = pd.read_csv(filepath, usecols=columns_needed, dtype=str)
    weather_df["DATE"] = pd.to_datetime(weather_df["DATE"], errors="coerce")
    weather_df["LATITUDE"] = pd.to_numeric(weather_df["LATITUDE"], errors="coerce")
    weather_df["LONGITUDE"] = pd.to_numeric(weather_df["LONGITUDE"], errors="coerce")
    hourly = weather_df[weather_df["REPORT_TYPE"].str.strip() == "FM-15"].copy()
    hourly["HourlyPrecipitation"] = pd.to_numeric(hourly["HourlyPrecipitation"].str.replace(
          "T", "0.001", regex=False), errors="coerce")
    hourly["temp_celsius"] = pd.to_numeric(hourly["HourlyDryBulbTemperature"], errors="coerce")
    hourly["HourlyWindSpeed"] = pd.to_numeric(hourly["HourlyWindSpeed"], errors="coerce")
    hourly["precip_type"] = (hourly["HourlyPresentWeatherType"].apply(classify_precip_type))

    return hourly[["STATION", "NAME", "LATITUDE", "LONGITUDE", "DATE", "HourlyPrecipitation", 
                    "temp_celsius", "HourlyWindSpeed", "precip_type"]]

In [20]:
# Loading the raw NOAA weather files required for the analysis period
files = glob.glob("../data/raw/LCD_*.csv")
print(f"Found {len(files)} weather files")

# Applying the load_weather_data function to each file
weather_dfs = []
for file in files:
    print(f"  Processing: {os.path.basename(file)}")
    df = load_weather_data(file)
    weather_dfs.append(df)

# Combining the processed weather files into a single DataFrame
weather_df = pd.concat(weather_dfs, ignore_index=True)
print(f"Combined Shape: {weather_df.shape}")

before_removing = weather_df.shape[0]

# Removing duplicate weather observations for the same station and timestamp
weather_df = weather_df.drop_duplicates(subset=["STATION", "DATE"], keep="first")

print(f"Duplicate rows removed: {before_removing - weather_df.shape[0]:,}")
print(f"\nShape after removing duplicates: {weather_df.shape}")

Found 8 weather files
  Processing: LCD_USW00014732_2024.csv
  Processing: LCD_USW00094789_2023.csv
  Processing: LCD_USW00014734_2023.csv
  Processing: LCD_USW00014734_2024.csv
  Processing: LCD_USW00094789_2024.csv
  Processing: LCD_USW00014732_2023.csv
  Processing: LCD_USW00094728_2024.csv
  Processing: LCD_USW00094728_2023.csv
Combined Shape: (70027, 9)
Duplicate rows removed: 8

Shape after removing duplicates: (70019, 9)


In [21]:
# Specifying the schema of the weather DataFrame 

weather_df = weather_df.astype({
    "STATION": "string",
    "NAME": "string",
    "LATITUDE": "float64",
    "LONGITUDE": "float64",
    "HourlyPrecipitation": "float64",
    "temp_celsius": "float64",
    "HourlyWindSpeed": "float64",
    "precip_type": "string"
})

weather_df["DATE"] = pd.to_datetime(weather_df["DATE"])


In [22]:
# Filtering weather observations to the six-month analysis period

before_rows, before_columns = weather_df.shape

start_date = pd.Timestamp("2023-11-01")
end_date = pd.Timestamp("2024-04-30 23:59:59")

weather_df = weather_df[
    (weather_df["DATE"] >= start_date) &
    (weather_df["DATE"] <= end_date)
].copy()

after_rows, after_columns = weather_df.shape

print(f"Rows before: {before_rows:,}")
print(f"Rows after:  {after_rows:,}")
print(f"Removed:     {before_rows - after_rows:,} ({(before_rows - after_rows) / before_rows * 100:.2f}%)")
print(f"Columns:     {after_columns}")

Rows before: 70,019
Rows after:  17,444
Removed:     52,575 (75.09%)
Columns:     9


In [23]:
# Checking for missing values in the dataset
missing_values = pd.DataFrame({
    "missing_count": weather_df.isna().sum(),
    "missing_percent": weather_df.isna().mean() * 100
})
print(missing_values)

                     missing_count  missing_percent
STATION                          0         0.000000
NAME                             0         0.000000
LATITUDE                         0         0.000000
LONGITUDE                        0         0.000000
DATE                             0         0.000000
HourlyPrecipitation            688         3.944050
temp_celsius                     1         0.005733
HourlyWindSpeed                332         1.903233
precip_type                      0         0.000000


In [24]:
# Saving the processed weather dataset

final_path = processed_dir / "weather_df.parquet"
weather_df.to_parquet(final_path, index=False)

weather_df = pd.read_parquet(final_path)

print(f"Rows:    {weather_df.shape[0]:,}")
print(f"Columns: {weather_df.shape[1]}")

Rows:    17,444
Columns: 9


## 3. Zone to Weather Station Matching

In [25]:
# Loading the NYC taxi zone shapefile 
zones_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip"
zones_zip = "../data/raw/taxi_zones.zip"
zones_dir = "../data/raw/taxi_zones"
zones_path = "../data/raw/taxi_zones/taxi_zones.shp"

if not os.path.exists(zones_dir):
    print("Downloading taxi zone shapefile")

    urllib.request.urlretrieve(zones_url, zones_zip)

    with zipfile.ZipFile(zones_zip, "r") as zip_ref:
        zip_ref.extractall("../data/raw")

zones = gpd.read_file(zones_path)

# Calculating the centroid of each zone
centroids = zones.copy()
centroids["geometry"] = centroids.geometry.centroid
centroids = centroids.to_crs(epsg=4326)  

centroids_final = centroids[["LocationID"]].copy()
centroids_final["lat"] = centroids.geometry.y
centroids_final["lon"] = centroids.geometry.x

print(f"Zone centroids computed: {len(centroids_final)} zones")


Zone centroids computed: 263 zones


In [26]:
# Matching each taxi zone to its nearest weather station using a BallTree algorithm 
# for efficient nearest-neighbour search
# Reference: 
# GeeksforGeeks, “Ball tree and KD tree algorithms,” GeeksforGeeks, Dec. 9, 2023. 
# https://www.geeksforgeeks.org/machine-learning/ball-tree-and-kd-tree-algorithms/.

station_df = (weather_df[["STATION", "NAME", "LATITUDE", "LONGITUDE"]
                         ].drop_duplicates().reset_index(drop=True))

station_coords_rad = np.radians(station_df[["LATITUDE", "LONGITUDE"]].values)

# Building the BallTree using haversine distance for geographic coordinates
station_tree = BallTree(station_coords_rad, metric="haversine")

zone_coords_rad = np.radians(centroids_final[["lat", "lon"]].values)
distances, indices = station_tree.query(zone_coords_rad, k=1)

zone_to_station = centroids_final.copy()
zone_to_station["nearest_station"] = station_df.iloc[indices[:, 0]]["STATION"].values

# Converting the haversine distances from radians to miles
zone_to_station["distance_to_station_miles"] = distances[:, 0] * 3958.8

print(f"Zones matched to nearest station: {len(zone_to_station)}")
print("\nZones matched to each station:")
print(zone_to_station["nearest_station"].value_counts())

print("\nDistance to nearest station (miles):")
print(zone_to_station["distance_to_station_miles"].describe())

# Creating the zone-to-station mapping for the join
zone_to_station_for_join = zone_to_station[["LocationID", "nearest_station"]].copy()
zone_to_station_spark = spark.createDataFrame(zone_to_station_for_join)


Zones matched to nearest station: 263

Zones matched to each station:
nearest_station
USW00094728    100
USW00014732     87
USW00094789     44
USW00014734     32
Name: count, dtype: Int64

Distance to nearest station (miles):
count    263.000000
mean       5.085839
std        2.677014
min        0.309428
25%        2.913580
50%        4.977791
75%        6.885443
max       11.988045
Name: distance_to_station_miles, dtype: float64


In [27]:
before_count = taxi_df.count()

# Assigning the nearest weather station to each taxi trip based on its pickup zone
taxi_df = taxi_df.join(
    F.broadcast(zone_to_station_spark),
    taxi_df.PULocationID == zone_to_station_spark.LocationID,
    "left"
).drop("LocationID")

before_count, taxi_df = shape_change(
    "Joining nearest weather station to each trip's pickup zone",
    before_count, taxi_df
)

# Verifying if any taxi trip could not be assigned a weather station
unmatched_station = taxi_df.filter(F.col("nearest_station").isNull()).count()
print(f"Trips with no matched weather station: {unmatched_station:,}")

Joining nearest weather station to each trip's pickup zone
  Before:  16,801,832
  After:   16,801,832
  Removed: 0 (0.00%)
  Columns: 10

Trips with no matched weather station: 0


In [28]:
# Converting the weather data to a Spark DataFrame and extracting the
# date and hour from each weather observation for joining with taxi trips
weather_spark = spark.createDataFrame(weather_df)

weather_join = (
    weather_spark
    .withColumnRenamed("DATE", "weather_datetime")
    .withColumn("weather_date", F.to_date("weather_datetime"))
    .withColumn("weather_hour", F.hour("weather_datetime"))
)

# Selecting one weather observation per station, date and hour
weather_window = Window.partitionBy(
    "STATION", "weather_date", "weather_hour"
).orderBy(
    F.when(F.minute("weather_datetime") == 51, 0).otherwise(1))

weather_row_before = weather_join.count()

weather_join = (
    weather_join
    .withColumn("weather_rank", F.row_number().over(weather_window))
    .filter(F.col("weather_rank") == 1)
    .drop("weather_rank")
)

weather_row_after, weather_join = shape_change(
    "Retain one hourly reading per station/date/hour",
    weather_row_before, weather_join
)


Retain one hourly reading per station/date/hour
  Before:  17,444
  After:   17,433
  Removed: 11 (0.06%)
  Columns: 11



## 4. Taxi and Weather Join

In [29]:
# Joining weather and taxi DataFrames 

before_count = taxi_df.count()

# Joining each taxi trip to the weather observation from its nearest station by
# pickup date and pickup hour
taxi_weather = taxi_df.join(
    F.broadcast(weather_join),
    (
        (taxi_df.nearest_station == weather_join.STATION) &
        (taxi_df.pickup_date == weather_join.weather_date) &
        (taxi_df.pickup_hour == weather_join.weather_hour)
    ),
    "left"
)

before_count, taxi_weather = shape_change(
    "Join taxi trips to hourly weather observations",
    before_count, taxi_weather,
    save_step=True, save_name="after_weather_join"   
)

# Verifying whether any taxi trips could not be matched to a weather observation
matched = taxi_weather.filter(F.col("weather_datetime").isNotNull()).count()
unmatched = before_count - matched
print(f"Trips with no weather match: {unmatched:,} ({unmatched/before_count*100:.2f}%)")

Join taxi trips to hourly weather observations
  Before:  16,801,832
  After:   16,801,832
  Removed: 0 (0.00%)
  Columns: 21



Trips with no weather match: 29,491 (0.18%)


In [30]:
# Removing taxi trips that could not be matched to a weather observation
taxi_weather = taxi_weather.filter(F.col("weather_datetime").isNotNull())

final_count, taxi_weather = shape_change("Removing unmatched rows", before_count, taxi_weather)

Removing unmatched rows
  Before:  16,801,832
  After:   16,772,341
  Removed: 29,491 (0.18%)
  Columns: 21



In [31]:
# Checking for missing values in weather variables to inform necessary imputation
print("Missing values in weather variables after join:")

for column_name in ["HourlyPrecipitation", "temp_celsius", "HourlyWindSpeed"]:
    missing = taxi_weather.filter(F.col(column_name).isNull()).count()
    percentage = missing / final_count * 100
    print(f"  {column_name}: {missing:,} missing ({percentage:.2f}%)")

Missing values in weather variables after join:


  HourlyPrecipitation: 348,081 missing (2.08%)


  temp_celsius: 0 missing (0.00%)


  HourlyWindSpeed: 1,338,737 missing (7.98%)


In [32]:
# Removing columns that are no longer required for subsequent analysis

taxi_weather = taxi_weather.drop("nearest_station", "STATION", "weather_date", "weather_hour")

print(f"Final Columns: {len(taxi_weather.columns)}")
print(taxi_weather.columns)

Final Columns: 17
['pickup_date', 'pickup_hour', 'time_of_day', 'week_day', 'trip_duration_min', 'trip_distance', 'trip_speed_mph', 'fare_amount', 'PULocationID', 'NAME', 'LATITUDE', 'LONGITUDE', 'weather_datetime', 'HourlyPrecipitation', 'temp_celsius', 'HourlyWindSpeed', 'precip_type']


In [33]:
# Saving the processed taxi-weather dataset

final_path = processed_dir / "taxi_weather.parquet"
taxi_weather.write.mode("overwrite").parquet(str(final_path))

taxi_weather = spark.read.parquet(str(final_path))  

print(f"Rows:    {taxi_weather.count():,}")
print(f"Columns: {len(taxi_weather.columns)}")

Rows:    16,772,341
Columns: 17


In [34]:
# Deleting intermediate parquet files created by the shape_change function

temp_files = [
    processed_dir / "after_weather_join.parquet"
]

for file in temp_files:
    if file.exists():
        shutil.rmtree(file)
        print(f"Deleted: {file}")

Deleted: ../data/processed/after_weather_join.parquet
